## Step 8 - Object detection (OpenImages V4), one JSON per video

Runs an **OpenImages V4** detector (TF-Hub) over every keyframe and writes
**one JSON file per video**, so re-runs only process missing videos/frames:

```
objects/<video_name>.json    # { "<frame_id>": { "<label>": score, ... }, ... }
```

`frame_id = <video_name>_<idx>` (e.g. `L21_V001_000080`) - the same join key as
`frames_meta.json`, so the merged output plugs straight into the graph layer
(`data_processing/08_build_graph.py` reads `backend/data/objects/objects_data.json`).

**Why OpenImages V4?** The official `objects-aic25-b1` detections use the same
600-label space ("Person", "Motorcycle", "Clothing", ...), so labels stay
consistent whether a scene got its objects from the official keyframes or from
this notebook - and `GET /graph/labels` keeps one vocabulary.

**Three detector choices** (`DETECTOR` below):
- `ssd`   - SSD MobileNet V2: ~15-25 img/s on a T4 -> ~1h for 55k frames. Default.
- `frcnn` - Faster R-CNN Inception ResNet V2 (what the official data used):
  much better recall on small objects but ~1 s/img -> needs several 12h sessions.
- `yolo11` - Ultralytics YOLO11 (COCO 80 labels, NOT OpenImages 600): rất nhanh
  (~30-60 img/s trên T4 với yolo11n), pretrain COCO. Label được viết hoa kiểu
  OpenImages cho subset trùng (Person, Car, Motorcycle, ...); một số class COCO
  không trùng vocab official (tv, remote, sports ball, ...) nên graph labels sẽ
  lệch nhẹ - cân nhắc nếu backend filter theo vocab OpenImages.

The loop is resumable per video AND per frame, checkpointed every
`CHECKPOINT_EVERY` frames, so just re-run the notebook to continue.

Input: the keyframe dataset from Step 3 (`Keyframes/<batch>/<video>/<idx>.webp`).
TensorFlow + tensorflow_hub are preinstalled on Kaggle - no pip needed.
Set **Accelerator = GPU T4 x2** (P100 works for TF, but T4 is faster here).

In [ ]:
import os
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image
from tqdm.auto import tqdm

# QUAN TRỌNG: đừng để TF chiếm hết VRAM (mặc định TF preallocate cả GPU). YOLO11
# (torch) chạy chung GPU -> nếu không giới hạn thì inference đầu tiên bị CUDA OOM
# và kernel bị restart.
_gpus = tf.config.experimental.list_physical_devices('GPU')
if _gpus:
    try:
        for _g in _gpus:
            tf.config.experimental.set_memory_growth(_g, True)
    except RuntimeError as _e:
        print('TF memory growth:', _e)

print('TF', tf.__version__, '| GPU:', _gpus)

In [ ]:
# Step 3 keyframes (nested: Keyframes/<batch>/<video_name>/<idx>.webp)
# LƯU Ý: trỏ vào thư mục KEYFRAME (ảnh .webp), KHÔNG phải SceneJSON.
KEYFRAMES_DIR    = '/kaggle/input/datasets/nguynlngnamanh/data-processing-aic-2026/keyframes_L21-25'
OUTPUT_DIR       = './objects_l21-25'          # one <video_name>.json per video
DETECTOR         = 'yolo11'                # 'ssd' (fast) | 'frcnn' (official-quality, slow)
                                       # | 'yolo11' (Ultralytics, COCO 80 labels, rất nhanh)
YOLO_MODEL       = 'yolo11x.pt'         # khi DETECTOR='yolo11': yolo11n/s/m/l/x (.pt)
SCORE_THRESHOLD  = 0.10                 # keep raw-ish; the graph build filters at 0.3
MAX_LABELS       = 30                   # per frame, best score per label
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 200                  # save the video's JSON every N frames

HUB_URLS = {
    'ssd':   'https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1',
    'frcnn': 'https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1',
}

# YOLO11 chạy trong subprocess worker -> kernel KHÔNG import torch/ultralytics
# -> không bao giờ bị crash/restart (kể cả khi Save Version chạy env sạch).
NUM_GPUS         = 1        # số GPU dùng song song (đổi 1 nếu chỉ có 1 GPU)
CONFIG_FILE      = '_yolo_config.json'
WORKER_SCRIPT    = '_yolo_worker.py'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Dump config + sinh worker script. YOLO sẽ chạy trong subprocess (cell main),
# nên cell này chỉ cần chạy 1 lần, kernel không nạp gì nặng.
_config = {
    'keyframes_dir': str(Path(KEYFRAMES_DIR)),
    'output_dir': str(Path(OUTPUT_DIR)),
    'yolo_model': YOLO_MODEL,
    'score_threshold': SCORE_THRESHOLD,
    'max_labels': MAX_LABELS,
    'image_exts': list(IMAGE_EXTS),
    'checkpoint_every': CHECKPOINT_EVERY,
}
with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
    json.dump(_config, f, ensure_ascii=False, indent=2)

WORKER_SRC = r'''import argparse
import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path

CONFIG_PATH = '_yolo_config.json'
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

KEYFRAMES_DIR    = cfg['keyframes_dir']
OUTPUT_DIR       = cfg['output_dir']
YOLO_MODEL       = cfg['yolo_model']
SCORE_THRESHOLD  = cfg['score_threshold']
MAX_LABELS       = cfg['max_labels']
IMAGE_EXTS       = tuple(cfg['image_exts'])
CHECKPOINT_EVERY = cfg['checkpoint_every']

LOCK_FILE        = '_yolo_install.lock'

os.makedirs(OUTPUT_DIR, exist_ok=True)


def ensure_deps():
    # Cài 1 lần, lock để 2 worker không cài đè nhau. Việc cài xảy ra trong process
    # worker (không phải kernel) -> kernel không bao giờ bị crash/restart.
    if importlib.util.find_spec('ultralytics') is not None:
        return
    while True:
        if importlib.util.find_spec('ultralytics') is not None:
            return
        if os.path.exists(LOCK_FILE):
            print('[worker] chờ worker khác cài ultralytics xong...', flush=True)
            time.sleep(10)
            continue
        try:
            fd = os.open(LOCK_FILE, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            continue
        os.close(fd)
        try:
            print('[worker] đang cài torch + ultralytics (chỉ 1 lần)...', flush=True)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'torch==2.5.1', 'torchvision==0.20.1',
                            '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                            'ultralytics'], check=True)
        finally:
            if os.path.exists(LOCK_FILE):
                os.remove(LOCK_FILE)
        return


def detect_one(path, model):
    results = model(str(path), conf=SCORE_THRESHOLD, verbose=False)[0]
    objs = {}
    for box in results.boxes:
        s = float(box.conf)
        if s < SCORE_THRESHOLD:
            continue
        label = results.names[int(box.cls)].title()
        if s > objs.get(label, 0.0):
            objs[label] = round(s, 4)
    return dict(sorted(objs.items(), key=lambda kv: -kv[1])[:MAX_LABELS])


def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'


def load_done(video_name):
    p = out_path(video_name)
    if p.exists():
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_done(video_name, data):
    with out_path(video_name).open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)


def build_video_index():
    root = Path(KEYFRAMES_DIR)
    videos = {}
    for img in root.rglob('*'):
        if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
            video_name = img.parent.name
            frame_id = f'{video_name}_{img.stem}'
            videos.setdefault(video_name, []).append((frame_id, img))
    for v in videos:
        videos[v].sort(key=lambda t: t[0])
    return videos


def load_model():
    ensure_deps()
    from ultralytics import YOLO
    model = YOLO(YOLO_MODEL)
    return model


def run_smoke():
    frames = sorted(p for p in Path(KEYFRAMES_DIR).rglob('*') if p.suffix.lower() in IMAGE_EXTS)
    if not frames:
        print(f'[worker] KHÔNG tìm thấy keyframe trong {KEYFRAMES_DIR}', flush=True)
        sys.exit(1)
    model = load_model()
    print('[worker] smoke frame:', frames[0], flush=True)
    t0 = time.time()
    objs = detect_one(frames[0], model)
    print(f'[worker] inference: {time.time() - t0:.2f}s', flush=True)
    print('[worker] ->', objs, flush=True)


def run_corpus(video_names):
    model = load_model()
    print(f'[worker] model ready: {YOLO_MODEL}', flush=True)
    videos = build_video_index()
    total_done = 0
    t_start = time.time()
    for video_name in video_names:
        frames = videos[video_name]
        data = load_done(video_name)
        todo = [(fid, p) for fid, p in frames if fid not in data]
        if not todo:
            total_done += len(frames)
            continue
        for n, (fid, p) in enumerate(todo, 1):
            try:
                data[fid] = detect_one(p, model)
            except Exception as e:
                print(f'[WARN] {fid}: {type(e).__name__}: {e}', flush=True)
                data[fid] = {}
            if n % CHECKPOINT_EVERY == 0:
                save_done(video_name, data)
        save_done(video_name, data)
        total_done += len(frames)
        _el = time.time() - t_start
        print(
            f'[worker] {video_name}: {len(frames)} frames | '
            f'cum {total_done} | {_el:.1f}s | {total_done / max(_el, 1e-6):.2f} img/s',
            flush=True,
        )
    _el = time.time() - t_start
    print(
        f'[worker] done {total_done} frames in {_el / 60:.1f} min '
        f'| {total_done / max(_el, 1e-6):.2f} img/s',
        flush=True,
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--gpu', type=int, default=-1)
    ap.add_argument('--videos', nargs='*')
    ap.add_argument('--smoke', action='store_true')
    args = ap.parse_args()
    if args.gpu >= 0:
        os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
    if args.smoke:
        run_smoke()
    else:
        run_corpus(sorted(args.videos))


if __name__ == '__main__':
    main()
'''
with open(WORKER_SCRIPT, 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('config ->', CONFIG_FILE, '| worker script ->', WORKER_SCRIPT)


In [ ]:
# Detector dùng chung cho mọi video.
# - ssd/frcnn: load TF-Hub ngay trong kernel.
# - yolo11:    KHÔNG load ở đây - worker subprocess sẽ tự load (cell main).
if DETECTOR == 'yolo11':
    print('YOLO11 chạy trong worker subprocess (cell main). Skip load detector ở kernel.')
else:
    # TF1-style hub modules: call qua .signatures['default'] với ảnh float32 [1,H,W,3] trong [0,1].
    detector = hub.load(HUB_URLS[DETECTOR]).signatures['default']
    print('detector ready:', DETECTOR, '(OpenImages V4 600 labels)')


In [ ]:
# Smoke test - 1 frame đầu.
# yolo11: chạy qua worker (subprocess) để không nạp torch vào kernel.
if DETECTOR == 'yolo11':
    cmd = [sys.executable, WORKER_SCRIPT, '--gpu', '0', '--smoke']
    r = subprocess.run(cmd, capture_output=True, text=True,
                       encoding='utf-8', errors='replace')
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit('smoke test failed')
else:
    _frames = sorted(p for p in Path(KEYFRAMES_DIR).rglob('*') if p.suffix in IMAGE_EXTS)
    if not _frames:
        raise SystemExit(f'Không tìm thấy keyframe trong: {KEYFRAMES_DIR}')
    _first = _frames[0]
    print(_first.name, '->', detect_objects(_first))


In [ ]:
# ==== YOLO11: chạy NUM_GPUS worker song song (mỗi GPU 1 worker) ====
if DETECTOR == 'yolo11':
    video_names = sorted({p.parent.name for p in Path(KEYFRAMES_DIR).rglob('*')
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTS})
    print(len(video_names), 'videos')
    chunks = [video_names[i::NUM_GPUS] for i in range(NUM_GPUS)]
    for g in range(NUM_GPUS):
        print(f'gpu{g}: {len(chunks[g])} videos')

    t_start = time.time()
    procs = []
    for gpu, vlist in enumerate(chunks):
        if not vlist:
            continue
        cmd = [sys.executable, WORKER_SCRIPT, '--gpu', str(gpu), '--videos', *vlist]
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, encoding='utf-8', errors='replace')
        procs.append(p)

    for p in procs:
        for line in p.stdout:
            print(line, end='', flush=True)
        rc = p.wait()
        if rc != 0:
            raise SystemExit(f'worker exited with code {rc}')
    print(f'\nall workers done in {(time.time() - t_start) / 60:.1f} min')
else:
    # ==== SSD / FRCNN: detect trong kernel như cũ ====
    video_dirs = sorted(d for d in Path(KEYFRAMES_DIR).glob('*/*') if d.is_dir())
    print(len(video_dirs), 'videos')
    for video_dir in tqdm(video_dirs, desc='videos'):
        video_name = video_dir.name
        out_path = Path(OUTPUT_DIR) / f'{video_name}.json'
        done = json.loads(out_path.read_text(encoding='utf-8')) if out_path.exists() else {}

        frames = sorted(p for p in video_dir.iterdir() if p.suffix in IMAGE_EXTS)
        todo = [(f'{video_name}_{p.stem}', p) for p in frames if f'{video_name}_{p.stem}' not in done]
        if not todo:
            continue

        for i, (frame_id, p) in enumerate(tqdm(todo, desc=video_name, leave=False), 1):
            try:
                done[frame_id] = detect_objects(p)
            except Exception as e:
                print(f'[WARN] {frame_id}: {type(e).__name__}: {e}')
                done[frame_id] = {}
            if i % CHECKPOINT_EVERY == 0:
                out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
        out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')

    print('done:', sum(1 for _ in Path(OUTPUT_DIR).glob('*.json')), 'video JSONs')


In [ ]:
# Merge the per-video files into the single file the graph build consumes.
merged = {}
for f in sorted(Path(OUTPUT_DIR).glob('*.json')):
    merged.update(json.loads(f.read_text(encoding='utf-8')))

with open('objects_data.json', 'w', encoding='utf-8') as fh:
    json.dump(merged, fh, ensure_ascii=False)

n_nonempty = sum(1 for v in merged.values() if v)
print(f'objects_data.json: {len(merged)} frames ({n_nonempty} with detections), '
      f'{os.path.getsize("objects_data.json") / 1e6:.1f} MB')

## Use with the backend

```text
objects_data.json -> backend/data/objects/objects_data.json
```

l

```bash
cd data_processing
python 08_build_graph.py            # + --knn to refresh similarity edges too
```

Check via the API: `GET /graph/frame/<frame_id>` should show `objects` for
frames the official keyframes never covered, and `GET /graph/labels` gets richer.